In [1]:
# !pip install datasets
!pip install transformers
!pip install torch

In [31]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score

In [32]:
train_path = "training_data/NLI/train.csv"
dev_path = "training_data/NLI/dev.csv"

In [33]:
train_df = pd.read_csv(train_path)
dev_df = pd.read_csv(dev_path)

train_dataset = Dataset.from_pandas(train_df)
dev_dataset = Dataset.from_pandas(dev_df)

print(train_dataset)

Dataset({
    features: ['premise', 'hypothesis', 'label'],
    num_rows: 24432
})


In [34]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [35]:
NEG_WORDS = {
    "not", "no", "never", "none", "nobody", "nothing", "neither",
    "nowhere", "hardly", "scarcely", "barely", "cannot", "can't", "won't", "isn't"
}

CONTRAST_WORDS = {
    "but", "however", "although", "though", "yet", "despite"
}

def extract_features(premise, hypothesis):
    p = premise.lower().split()
    h = hypothesis.lower().split()

    p_set = set(p)
    h_set = set(h)

    overlap = len(p_set & h_set)
    union = len(p_set | h_set)
    jaccard = overlap / union if union > 0 else 0.0

    p_len = len(p)
    h_len = len(h)
    len_diff = abs(p_len - h_len)

    neg_p = sum(word in NEG_WORDS for word in p)
    neg_h = sum(word in NEG_WORDS for word in h)
    neg_diff = abs(neg_p - neg_h)

    contrast_p = sum(word in CONTRAST_WORDS for word in p)
    contrast_h = sum(word in CONTRAST_WORDS for word in h)

    hyp_in_prem = int(" ".join(h) in " ".join(p))

    return [
        float(overlap),
        float(jaccard),
        float(p_len),
        float(h_len),
        float(len_diff),
        float(neg_p),
        float(neg_h),
        float(neg_diff),
        float(contrast_p),
        float(contrast_h),
        float(hyp_in_prem),
    ]

In [36]:
MAX_LEN = 128

def preprocess(example):
    encoded = tokenizer(
        example["premise"],
        example["hypothesis"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

    features = extract_features(example["premise"], example["hypothesis"])

    encoded["features"] = features
    encoded["label"] = int(example["label"])
    return encoded

In [37]:
train_dataset = train_dataset.map(preprocess)
dev_dataset = dev_dataset.map(preprocess)

Map:   0%|          | 0/24432 [00:00<?, ? examples/s]

Map:   0%|          | 0/6736 [00:00<?, ? examples/s]

In [38]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "features", "label"]
)

dev_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "features", "label"]
)

In [39]:
class MultiExpertNLI(nn.Module):
    def __init__(self, model_name="bert-base-uncased", num_labels=2, feature_dim=11):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        # Expert 1: semantic expert
        self.semantic_classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_labels)
        )

        # Expert 2: lexical expert
        self.lexical_expert = nn.Sequential(
            nn.Linear(feature_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, num_labels)
        )

        # Expert 3: logic expert
        self.logic_expert = nn.Sequential(
            nn.Linear(feature_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, num_labels)
        )

        # Gating network decides expert importance
        self.gating_network = nn.Sequential(
            nn.Linear(hidden_size + feature_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 3)
        )

        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask, features, labels=None):
        # BERT encoding
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls = outputs.last_hidden_state[:, 0, :]   # [CLS]

        # Expert outputs
        semantic_logits = self.semantic_classifier(cls)
        lexical_logits = self.lexical_expert(features.float())
        logic_logits = self.logic_expert(features.float())

        # Gating input combines deep + symbolic info
        gate_input = torch.cat([cls, features.float()], dim=1)
        gate_logits = self.gating_network(gate_input)
        gate_weights = torch.softmax(gate_logits, dim=1)

        # Weighted combination
        final_logits = (
            gate_weights[:, 0:1] * semantic_logits +
            gate_weights[:, 1:2] * lexical_logits +
            gate_weights[:, 2:3] * logic_logits
        )

        loss = None
        if labels is not None:
            loss = self.loss_fn(final_logits, labels)

        return {
            "loss": loss,
            "logits": final_logits
        }

In [40]:
model = MultiExpertNLI(
    model_name="bert-base-uncased",
    num_labels=2,
    feature_dim=11
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [41]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    macro_f1 = f1_score(labels, preds, average="macro")
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "macro_f1": macro_f1
    }

In [42]:
training_args = TrainingArguments(
    output_dir="results_multi_expert",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_dir="logs_multi_expert",
    report_to="none",
    disable_tqdm=False,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [43]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics
)

In [45]:
trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time,Accuracy,Macro F1
1,0.432906,0.422290,0.003500,0.808789,0.808750
2,0.274270,0.480300,0.003500,0.826010,0.825388
3,0.145302,0.704970,0.003500,0.828236,0.828005


TrainOutput(global_step=4581, training_loss=0.31155861124048273, metrics={'train_runtime': 421.919, 'train_samples_per_second': 173.721, 'train_steps_per_second': 10.858, 'total_flos': 0.0, 'train_loss': 0.31155861124048273, 'epoch': 3.0})

In [46]:
trainer.evaluate()

{'eval_loss': 0.7049702405929565,
 'eval_model_preparation_time': 0.0035,
 'eval_accuracy': 0.8282363420427553,
 'eval_macro_f1': 0.8280050794604169,
 'eval_runtime': 11.8589,
 'eval_samples_per_second': 568.011,
 'eval_steps_per_second': 35.501,
 'epoch': 3.0}

In [30]:
torch.save(model.state_dict(), "model.pt")